In [1]:
import pandas as pd
import sqlite3

print("1. Loading raw data... (Reading 500,000 rows to save memory)")
# Read a chunk of the CSV. Make sure the path matches your folder structure!
file_path = '../data/raw/accepted_2007_to_2018Q4.csv'
df = pd.read_csv(file_path, nrows=500000, low_memory=False)

# Filter out "Current" loans to avoid Data Leakage
valid_statuses = ['Fully Paid', 'Charged Off']
df = df[df['loan_status'].isin(valid_statuses)].copy()

# Create binary target (1 = Default/Charged Off, 0 = Paid)
df['target'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)

# Select only pre-loan columns (Business Rule: No Data Leakage)
columns_to_keep = [
    'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'purpose',
    'dti', 'open_acc', 'total_acc', 'pub_rec', 'target'
]
df = df[columns_to_keep]

print("2. Saving clean data to SQLite database...")
# Connect to SQLite (this creates the file in your processed folder)
db_path = '../data/processed/lending_club.db'
conn = sqlite3.connect(db_path)
df.to_sql('loans', conn, if_exists='replace', index=False)

print("3. Running Business SQL Queries...\n")

# --- QUERY 1: Overall Default Rate ---
query1 = """
SELECT 
    COUNT(*) as total_loans, 
    ROUND(AVG(target) * 100, 2) as overall_default_rate_pct 
FROM loans;
"""
print("--- Query 1: Overall Portfolio Health ---")
print(pd.read_sql_query(query1, conn), "\n")


# --- QUERY 2: Default Rate by Grade ---
query2 = """
SELECT 
    grade, 
    COUNT(*) as total_loans, 
    ROUND(AVG(target) * 100, 2) as default_rate_pct 
FROM loans 
GROUP BY grade 
ORDER BY grade;
"""
print("--- Query 2: Risk Grade Performance ---")
print(pd.read_sql_query(query2, conn), "\n")


# --- QUERY 3: Interest Rates vs Defaults ---
query3 = """
SELECT 
    CASE WHEN target = 1 THEN 'Defaulter' ELSE 'Paid Off' END as loan_outcome,
    COUNT(*) as total_loans,
    ROUND(AVG(int_rate), 2) as avg_interest_rate 
FROM loans 
GROUP BY target;
"""
print("--- Query 3: Interest Rate Impact ---")
print(pd.read_sql_query(query3, conn), "\n")


# --- QUERY 4: Riskiest Loan Purposes ---
query4 = """
SELECT 
    purpose, 
    COUNT(*) as total_loans, 
    ROUND(AVG(target) * 100, 2) as default_rate_pct 
FROM loans 
GROUP BY purpose 
HAVING COUNT(*) > 1000 
ORDER BY default_rate_pct DESC 
LIMIT 5;
"""
print("--- Query 4: Top 5 Riskiest Loan Purposes ---")
print(pd.read_sql_query(query4, conn), "\n")


# --- QUERY 5: Employment Length Risk (Window Function) ---
query5 = """
SELECT 
    emp_length,
    COUNT(*) as total_loans,
    ROUND(AVG(target) * 100, 2) as default_rate_pct,
    RANK() OVER (ORDER BY AVG(target) DESC) as risk_rank
FROM loans
WHERE emp_length IS NOT NULL
GROUP BY emp_length
ORDER BY risk_rank;
"""
print("--- Query 5: Employment Length Risk Ranking ---")
print(pd.read_sql_query(query5, conn), "\n")

# Close the database connection
conn.close()
print("Phase 2 Complete! Database closed securely.")

1. Loading raw data... (Reading 500,000 rows to save memory)
2. Saving clean data to SQLite database...
3. Running Business SQL Queries...

--- Query 1: Overall Portfolio Health ---
   total_loans  overall_default_rate_pct
0       391164                     20.15 

--- Query 2: Risk Grade Performance ---
  grade  total_loans  default_rate_pct
0     A        75359              5.59
1     B       111201             13.06
2     C       108792             22.43
3     D        56162             32.25
4     E        29331             41.80
5     F         8394             50.92
6     G         1925             54.08 

--- Query 3: Interest Rate Impact ---
  loan_outcome  total_loans  avg_interest_rate
0     Paid Off       312340              11.77
1    Defaulter        78824              15.00 

--- Query 4: Top 5 Riskiest Loan Purposes ---
              purpose  total_loans  default_rate_pct
0      small_business         3266             28.60
1               house         1601             